# Model Exploration

Explore Qwen3.5-0.8B model configuration, GPU memory use, and numerical reasoning on a GSM8K validation example.

In [0]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GiB")


## Inspect model architecture and configuration

Read the local model configuration without loading the model weights.

In [0]:
from pathlib import Path
from collections import Counter
from transformers import AutoConfig

STORAGE_DIR = Path("/Volumes/workspace/default/numreason")
DATA_DIR = STORAGE_DIR / "data_split"
MODEL_DIR = STORAGE_DIR / "models" / "Qwen3.5-0.8B"
SEED = 189

if not (MODEL_DIR / "config.json").is_file():
    raise FileNotFoundError(f"Model configuration not found at {MODEL_DIR}. Run download_data.py first.")

config = AutoConfig.from_pretrained(str(MODEL_DIR))
text_config = getattr(config, "text_config", config)

fields = {
    "Model type": config.model_type,
    "Text model type": getattr(text_config, "model_type", "N/A"),
    "Hidden size": getattr(text_config, "hidden_size", "N/A"),
    "Number of layers": getattr(text_config, "num_hidden_layers", "N/A"),
    "Attention heads": getattr(text_config, "num_attention_heads", "N/A"),
    "Key/value heads": getattr(text_config, "num_key_value_heads", "N/A"),
    "Vocabulary size": getattr(text_config, "vocab_size", "N/A"),
    "Intermediate size": getattr(text_config, "intermediate_size", "N/A"),
    "Maximum position embeddings": getattr(text_config, "max_position_embeddings", "N/A"),
}

for name, value in fields.items():
    print(f"{name}: {value}")

layer_types = getattr(text_config, "layer_types", None)
if layer_types is not None:
    print("Layer types:", dict(Counter(layer_types)))


## Load the model and inspect parameters

Load Qwen3.5-0.8B and examine its parameter counts and GPU memory usage.

In [0]:
from transformers import AutoProcessor, AutoModelForMultimodalLM

if not torch.cuda.is_available():
    raise RuntimeError("Switch to Databricks Serverless GPU before loading Qwen3.5-0.8B.")

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.cuda.reset_peak_memory_stats()

processor = AutoProcessor.from_pretrained(str(MODEL_DIR))
model = AutoModelForMultimodalLM.from_pretrained(
    str(MODEL_DIR),
    dtype="auto",
    device_map="auto",
)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model: {MODEL_DIR.name}")
print(f"Total parameters (including vision components): {total_params:,}")
print(f"Trainable parameters before fine-tuning setup: {trainable_params:,}")
print(f"Total parameters in billions: {total_params / 1e9:.3f}B")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB")
print(f"GPU peak memory allocated: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GiB")


## Test Qwen on a GSM8K validation example

Generate a step-by-step solution and compare the final numerical answer with the reference. This is an exploratory test, not a benchmark.

In [0]:
import re
from decimal import Decimal, InvalidOperation
from datasets import load_from_disk

if not (DATA_DIR / "dataset_dict.json").is_file():
    raise FileNotFoundError(f"Processed dataset not found at {DATA_DIR}.")


def extract_final_number(text):
    matches = re.findall(r"####\s*\$?\s*([-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?)", text)
    if not matches:
        return None
    try:
        return Decimal(matches[-1].replace(",", ""))
    except InvalidOperation:
        return None


example = load_from_disk(str(DATA_DIR))["validation"][0]

prompt = (
    "Solve the following math problem step by step. "
    "End with a line containing #### followed by the final numerical answer.\n\n"
    f"{example['question']}"
)

messages = [{"role": "user", "content": prompt}]

inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=False,
    )

response = processor.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True,
)

reference_number = extract_final_number(example["answer"])
predicted_number = extract_final_number(response)

print("QUESTION:")
print(example["question"])
print("\nMODEL RESPONSE:")
print(response)
print("\nEVALUATION:")
print(f"Reference answer: {reference_number}")
print(f"Predicted answer: {predicted_number}")
print(f"Final answer parsed: {predicted_number is not None}")
print(f"Correct final answer: {predicted_number is not None and predicted_number == reference_number}")
